# LeRobot format

> Read LeRobot Hub datasets (`lerobot://<repo>`) through the `Dataset` API (stable-worldmodel's `LeRobotAdapter`).

In [ ]:
#| default_exp data.lerobot

In [ ]:
#| hide
from nbdev.showdoc import *

Read-only, as in stable-worldmodel: the `lerobot://` scheme wraps lerobot's
`LeRobotDataset` (needs the `lerobot` package, Python 3.12+). The primary camera becomes
`pixels`, `action` stays `action` and `observation.state` becomes `proprio`
(`key_aliases` adds or renames columns); `ep_idx` / `step_idx` are added.

LeRobot datasets are single-agent. With `add_agent_axis=True` (default) every column but
`ep_idx` / `step_idx` gets a 1-agent axis after time, laid out like stable-marl's own data:
`pixels` `(T, 1, H, W, C)` (channel-last, as per-agent images are), `proprio`
`(T, 1, D)`, and the `action` column `(N, 1, D)` (clips flatten it per step, as every
`Dataset`: `(num_steps, 1 * D)`), so the datasets plug into multi-agent code with one agent. `add_agent_axis=False` gives stable-worldmodel's layout.

In [ ]:
#| export
# Adapted from stable-worldmodel's data package (https://github.com/galilai-group/stable-worldmodel):
#
# MIT License
#
# Copyright (c) 2026 GalilAI-group
#
# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so, subject to the following conditions:
#
# The above copyright notice and this permission notice shall be included in all
# copies or substantial portions of the Software.
#
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
# SOFTWARE.

from __future__ import annotations

import sys
from pathlib import Path
from typing import Any, Callable, Mapping

import numpy as np

from stable_marl.data.dataset import Dataset
from stable_marl.data.format import Format, register_format

_SCHEME = 'lerobot://'

In [ ]:
#| export
def _import_lerobot_hub_dataset() -> type:
    "lerobot's `LeRobotDataset` (imported lazily)."
    if sys.version_info < (3, 12):
        raise ImportError("stable_marl.data.LeRobotAdapter requires Python 3.12+ because the official lerobot "
                          "package is only available on Python 3.12+.")
    try:
        from lerobot.datasets.lerobot_dataset import LeRobotDataset as LerobotHubDataset
    except ImportError as exc:
        raise ImportError("stable_marl.data.LeRobotAdapter requires the optional lerobot dependency: "
                          "pip install lerobot") from exc
    return LerobotHubDataset


def _scalarize(value: Any) -> Any:
    if type(value).__module__.startswith('torch'):
        return value.item() if value.ndim == 0 else value.detach().cpu().numpy()
    if isinstance(value, np.ndarray):
        return value.item() if value.ndim == 0 else value
    return value


def _column_to_numpy(column: Any) -> np.ndarray:
    if type(column).__module__.startswith('torch'):
        return column.detach().cpu().numpy()
    if isinstance(column, np.ndarray):
        return column
    if isinstance(column, list):
        return np.asarray([_scalarize(v) for v in column])
    return np.asarray(column)


class LeRobotAdapter(Dataset):
    "Wraps lerobot's `LeRobotDataset` with the stable-marl `Dataset` API (see above)."
    _SYNTHETIC_COLUMNS = {'ep_idx', 'step_idx'}

    def __init__(
        self,
        repo_id: str,
        root: str | Path | None = None,
        episodes: list[int] | None = None,
        frameskip: int = 1,
        num_steps: int = 1,
        transform: Callable[[dict], dict] | None = None,
        keys_to_load: list[str] | None = None,
        keys_to_cache: list[str] | None = None,
        primary_camera_key: str | None = None,
        key_aliases: dict[str, str] | None = None,
        add_agent_axis: bool = True,
        **lerobot_kwargs: Any):
        LerobotHubDataset = _import_lerobot_hub_dataset()
        self._hub_dataset_cls, self._lerobot_kwargs = LerobotHubDataset, dict(lerobot_kwargs)
        self.add_agent_axis = add_agent_axis
        self.dataset = LerobotHubDataset(repo_id=repo_id, root=root, episodes=episodes, image_transforms=None,
                                         delta_timestamps=None, **lerobot_kwargs)
        self.repo_id, self.root, self.episodes = repo_id, Path(root) if root is not None else None, episodes
        native_keys = self._get_native_keys()
        self._camera_keys = self.dataset.meta.camera_keys
        self._fps = self._get_fps()
        self._primary_camera_key = self._resolve_primary_camera(primary_camera_key, native_keys)
        self._native_to_alias = self._build_alias_map(native_keys, key_aliases)
        self._alias_to_native = {alias: native for native, alias in self._native_to_alias.items()}
        self._full_columns: dict[str, np.ndarray] = {}
        self._cache: dict[str, np.ndarray] = {}
        self._window_datasets: dict = {}
        local_episode_index, step_idx, lengths, offsets, absolute_episode_ids = \
            self._build_episode_metadata(self._get_native_column('episode_index'))
        self._absolute_episode_ids = absolute_episode_ids
        self._cache['ep_idx'], self._cache['step_idx'] = local_episode_index, step_idx
        if keys_to_load is None:
            keys_to_load = list(self._native_to_alias.values()) + ['ep_idx', 'step_idx']
        self._keys = list(dict.fromkeys(keys_to_load))
        for key in keys_to_cache or []:
            self._cache[key] = self._materialize_column(key)
        super().__init__(lengths, offsets, frameskip, num_steps, transform)

    @property
    def column_names(self) -> list[str]:
        return self._keys

    def _get_native_keys(self) -> list[str]:
        features = self.dataset.features
        if not isinstance(features, Mapping):
            raise TypeError("LeRobot dataset features must be a mapping of column names.")
        return list(features.keys())

    def _resolve_primary_camera(self, primary_camera_key: str | None, native_keys: list[str]) -> str | None:
        if primary_camera_key is None and len(self._camera_keys) > 1:
            raise ValueError("LeRobotAdapter requires `primary_camera_key` when multiple cameras are available.")
        if primary_camera_key is not None:
            if primary_camera_key not in native_keys:
                raise KeyError(f"Primary camera key '{primary_camera_key}' not found in LeRobot dataset.")
            return primary_camera_key
        return next((k for k in self._camera_keys if k in native_keys), None)

    def _get_fps(self) -> float:
        info = self.dataset.meta.info
        if isinstance(info, dict) and 'fps' in info:
            return float(info['fps'])
        if hasattr(info, 'fps'):
            return float(info.fps)
        raise ValueError("LeRobot dataset metadata must expose `meta.info.fps`.")

    def _build_alias_map(self, native_keys: list[str], key_aliases: dict[str, str] | None) -> dict[str, str]:
        aliases: dict[str, str] = {}
        if self._primary_camera_key is not None:
            aliases[self._primary_camera_key] = 'pixels'
        if 'action' in native_keys:
            aliases['action'] = 'action'
        if 'observation.state' in native_keys:
            aliases['observation.state'] = 'proprio'
        for native, alias in (key_aliases or {}).items():
            if native not in native_keys:
                raise KeyError(f"Key alias source '{native}' not found in LeRobot dataset.")
            aliases[native] = alias
        return aliases

    def _build_episode_metadata(self, absolute_episode_index: np.ndarray):
        abs_ids = absolute_episode_index.astype(np.int64)
        unique_abs, first_idx = np.unique(abs_ids, return_index=True)
        absolute_episode_ids = unique_abs[np.argsort(first_idx)]
        counts = np.array([(abs_ids == ep).sum() for ep in absolute_episode_ids], dtype=np.int64)
        local_map = {int(abs_id): idx for idx, abs_id in enumerate(absolute_episode_ids)}
        local_episode_index = np.array([local_map[int(a)] for a in abs_ids], dtype=np.int64)
        step_idx = np.empty_like(local_episode_index)
        for local_ep in range(len(absolute_episode_ids)):
            mask = local_episode_index == local_ep
            step_idx[mask] = np.arange(mask.sum(), dtype=np.int64)
        offsets = np.zeros(len(counts), dtype=np.int64)
        if len(counts) > 1:
            offsets[1:] = np.cumsum(counts[:-1])
        return local_episode_index, step_idx, counts, offsets, absolute_episode_ids.astype(np.int64)

    def _get_native_column(self, native_key: str) -> np.ndarray:
        if native_key not in self._full_columns:
            self._full_columns[native_key] = _column_to_numpy(self.dataset.hf_dataset[native_key])
        return self._full_columns[native_key]

    def _time_offsets(self, indices: tuple[int, ...]) -> list[float]:
        return [float(idx) / self._fps for idx in indices]

    def _window_dataset(self, observation_indices: tuple[int, ...], action_indices: tuple[int, ...]) -> Any:
        cache_key = (observation_indices, action_indices)
        if cache_key not in self._window_datasets:
            delta_timestamps = {}
            for key in self._keys:
                native_key = None if key in self._SYNTHETIC_COLUMNS else self._alias_to_native.get(key)
                if native_key is not None:
                    delta_timestamps[native_key] = self._time_offsets(action_indices if key == 'action' else observation_indices)
            self._window_datasets[cache_key] = self._hub_dataset_cls(
                repo_id=self.repo_id, root=self.root, episodes=self.episodes, image_transforms=None,
                delta_timestamps=delta_timestamps or None, **self._lerobot_kwargs)
        return self._window_datasets[cache_key]

    def _materialize_column(self, key: str) -> np.ndarray:
        if key in self._cache:
            return self._cache[key]
        native_key = self._alias_to_native.get(key)
        if native_key is None:
            raise KeyError(f"Unknown LeRobot adapter column '{key}'.")
        if native_key in self._camera_keys:
            raise KeyError(f"'{key}' cannot be materialized as a full array because it is image/video-backed.")
        return self._get_native_column(native_key)

    def _agent_axis(self, key: str, data):
        "A column with a 1-agent axis after time (images channel-last, as per-agent images are)."
        if not self.add_agent_axis or key in self._SYNTHETIC_COLUMNS:
            return data
        if self._alias_to_native.get(key) in self._camera_keys and data.ndim == 4 and data.shape[1] in (1, 3):
            data = data.permute(0, 2, 3, 1)                   # (T, C, H, W) -> (T, H, W, C)
        return data.unsqueeze(1) if hasattr(data, 'unsqueeze') else np.expand_dims(data, 1)

    def _load_slice(self, ep_idx: int, start: int, end: int) -> dict:
        import torch
        g_start, length = int(self.offsets[ep_idx] + start), int(end - start)
        obs_indices, action_indices = tuple(range(0, length, self.frameskip)), tuple(range(length))
        row = dict(self._window_dataset(obs_indices, action_indices)[g_start])
        steps: dict[str, Any] = {}
        for key in self._keys:
            if key == 'ep_idx':
                data = torch.full((len(obs_indices),), int(self._cache['ep_idx'][g_start]), dtype=torch.int64)
            elif key == 'step_idx':
                data = torch.as_tensor([start + idx for idx in obs_indices], dtype=torch.int64)
            else:
                data = row[self._alias_to_native[key]]
                if torch.is_tensor(data) and not self.add_agent_axis and data.ndim == 4 and data.shape[-1] in (1, 3):
                    data = data.permute(0, 3, 1, 2)
            steps[key] = self._agent_axis(key, data) if torch.is_tensor(data) else data
        return self.transform(steps) if self.transform else steps

    def get_col_data(self, col: str) -> np.ndarray:
        return self._agent_axis(col, self._materialize_column(col))

    def get_row_data(self, row_idx: int | list[int]) -> dict:
        out = {}
        for col in self._keys:
            try:
                data = self.get_col_data(col)
            except KeyError:
                continue
            out[col] = data[row_idx]
        return out


@register_format
class LeRobot(Format):
    "LeRobot Hub datasets, as `lerobot://<repo_id>` (read-only)."
    name = 'lerobot'

    @classmethod
    def detect(cls, path) -> bool:
        return isinstance(path, str) and path.startswith(_SCHEME)

    @classmethod
    def open_reader(cls, path, **kwargs):
        return LeRobotAdapter(path[len(_SCHEME):] if path.startswith(_SCHEME) else path, **kwargs)

### Tests

With a stand-in for lerobot's `LeRobotDataset` (the package needs Python 3.12+): two episodes, one camera, windows over `delta_timestamps` clamped to the episode, as lerobot does.

In [ ]:
import torch
from types import SimpleNamespace
from fastcore.test import test_fail
from stable_marl.data import load_dataset, GoalDataset

class FakeHubDataset:
    "lerobot's LeRobotDataset in miniature: 2 episodes (5 and 3 frames), camera 'observation.image' (C, H, W) floats."
    def __init__(self, repo_id, root=None, episodes=None, image_transforms=None, delta_timestamps=None, **kw):
        self.ep = np.array([7] * 5 + [9] * 3)                 # absolute episode ids
        n = len(self.ep)
        self.cols = {'episode_index': self.ep, 'action': np.arange(n * 2, dtype=np.float32).reshape(n, 2),
                     'observation.state': np.arange(n * 3, dtype=np.float32).reshape(n, 3),
                     'observation.image': np.random.default_rng(0).random((n, 3, 4, 6)).astype(np.float32)}
        self.features = {k: {} for k in self.cols}
        self.meta = SimpleNamespace(camera_keys=['observation.image'], info={'fps': 10})
        self.hf_dataset = {k: v for k, v in self.cols.items() if k != 'observation.image'}
        self.delta = delta_timestamps

    def __getitem__(self, i):
        same = np.flatnonzero(self.ep == self.ep[i])
        out = {}
        for k, v in self.cols.items():
            if self.delta and k in self.delta:
                idx = np.clip([i + round(t * 10) for t in self.delta[k]], same[0], same[-1])
                out[k] = torch.as_tensor(v[idx])
            else:
                out[k] = torch.as_tensor(v[i])
        return out

_real_import = _import_lerobot_hub_dataset
_import_lerobot_hub_dataset = lambda: FakeHubDataset
try:
    ds = load_dataset('lerobot://someone/robot', num_steps=3)
    assert isinstance(ds, LeRobotAdapter) and ds.lengths.tolist() == [5, 3]
    assert set(ds.column_names) == {'pixels', 'action', 'proprio', 'ep_idx', 'step_idx'}
    item = ds[0]
    assert item['pixels'].shape == (3, 1, 4, 6, 3) and item['action'].shape == (3, 1 * 2)   # 1 agent (actions per step, as Dataset)
    assert item['proprio'].shape == (3, 1, 3) and item['step_idx'].tolist() == [0, 1, 2]
    assert ds.get_col_data('action').shape == (8, 1, 2) and ds.get_col_data('ep_idx').tolist() == [0] * 5 + [1] * 3
    assert ds[len(ds) - 1]['ep_idx'].tolist() == [1, 1, 1]
    flat = LeRobotAdapter('someone/robot', num_steps=2, add_agent_axis=False)           # stable-worldmodel's layout
    assert flat[0]['pixels'].shape == (2, 3, 4, 6) and flat[0]['action'].shape == (2, 2)
    g = GoalDataset(ds, goal_probabilities=(0.0, 1.0, 0.0, 0.0), seed=0)
    assert g[0]['goal_pixels'].shape == (1, 1, 4, 6, 3)
    test_fail(lambda: LeRobotAdapter('someone/robot', key_aliases={'nope': 'x'}), contains='nope')
finally:
    _import_lerobot_hub_dataset = _real_import
if sys.version_info < (3, 12):
    test_fail(lambda: LeRobotAdapter('someone/robot'), contains='Python 3.12')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()